In [1]:
import pandas as pd

In [2]:
train_df = pd.read_csv("./cnn_dailymail/train.csv")
# train_df = train_df.drop(columns=["id"])
train_df["id"] = train_df.index
train_df = train_df.rename(columns={"highlights":"summary", "article":"text"})
train_df.head(2)

,id,text,summary
0,0,By . Associated Press . PUBLISHED: . 14:11 EST...,"Bishop John Folda, of North Dakota, is taking ..."
1,1,(CNN) -- Ralph Mata was an internal affairs li...,Criminal complaint: Cop used his role to help ...


In [3]:
# test_df = pd.read_csv("./cnn_dailymail/validation.csv")
# test_df = test_df.drop(columns=["id"])
# test_df = test_df.rename(columns={"highlights":"summary"})
# test_df.head(2)

In [4]:
import pandas as pd
from neo4j import GraphDatabase
from langchain_ollama import OllamaLLM
import json, re, time

# ── Setup ──────────────────────────────────────────────────
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "12345678"))
llm = OllamaLLM(model="llama3.2", format="json", temperature=0, num_predict=2048)

# ── Prompt with one-shot example (fixes JSON issues) ──────
PROMPT = """Extract all entities and relationships from this text.

Return ONLY valid JSON in this EXACT format:
{{"entities":[{{"name":"Apple","type":"ORG"}},{{"name":"Tim Cook","type":"PERSON"}},{{"name":"Cupertino","type":"LOCATION"}}],"relationships":[{{"source":"Tim Cook","type":"CEO_OF","target":"Apple"}},{{"source":"Apple","type":"HEADQUARTERED_IN","target":"Cupertino"}}]}}

Rules:
- "name" and "type" are strings
- "source" and "target" are strings (NEVER arrays), must match an entity name exactly
- relationship "type" is UPPERCASE_WITH_UNDERSCORES
- Extract every fact

Text:
{text}"""


# ── Chunking (kept, improved with overlap) ────────────────
def chunk_text(text, max_chars=2000, overlap=200):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks, current = [], ""
    for sent in sentences:
        if len(current) + len(sent) > max_chars and current:
            chunks.append(current.strip())
            # keep last `overlap` chars for context continuity
            current = current[-overlap:] + " " + sent
        else:
            current = (current + " " + sent) if current else sent
    if current.strip():
        chunks.append(current.strip())
    return chunks if chunks else [text]


# ── Robust JSON parsing ──────────────────────────────────
def parse_json(raw):
    try:
        return json.loads(raw)
    except Exception:
        pass
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except Exception:
            pass
    return None


# ── Validate & clean one chunk's output ──────────────────
def clean(data):
    if not data:
        return [], []

    entities, names = [], set()
    for e in data.get("entities", []):
        if not isinstance(e, dict):
            continue
        name = str(e.get("name", "")).strip()
        if not name:
            continue
        etype = re.sub(r"[^A-Z0-9_]", "_", str(e.get("type", "THING")).upper())
        entities.append({"name": name, "type": etype})
        names.add(name)

    rels = []
    for r in data.get("relationships", []):
        if not isinstance(r, dict):
            continue
        s = r.get("source", "")
        t = r.get("target", "")
        rtype = r.get("type") or r.get("relation") or "RELATED_TO"
        rtype = re.sub(r"[^A-Z0-9_]", "_", str(rtype).upper())

        # handle LLM returning lists despite instructions
        sources = s if isinstance(s, list) else [s]
        targets = t if isinstance(t, list) else [t]
        for src in sources:
            for tgt in targets:
                src, tgt = str(src).strip(), str(tgt).strip()
                if src in names and tgt in names and src != tgt:
                    rels.append({"source": src, "target": tgt, "type": rtype})

    return entities, rels


# ── Extract from one chunk (with retry) ──────────────────
def extract_chunk(chunk):
    for _ in range(2):
        try:
            raw = llm.invoke(PROMPT.format(text=chunk))
            data = parse_json(raw)
            ents, rels = clean(data)
            if ents:
                return ents, rels
        except Exception:
            continue
    return [], []


# ── Merge entities across chunks (dedup by name) ─────────
def extract_kg(text):
    chunks = chunk_text(text)
    all_ents, all_rels = [], []

    for chunk in chunks:
        ents, rels = extract_chunk(chunk)
        all_ents.extend(ents)
        all_rels.extend(rels)

    # deduplicate entities by lowercase name
    seen, unique = set(), []
    for e in all_ents:
        key = e["name"].lower()
        if key not in seen:
            seen.add(key)
            unique.append(e)

    # deduplicate relationships
    seen_rels, unique_rels = set(), []
    entity_names = {e["name"] for e in unique}
    for r in all_rels:
        if r["source"] in entity_names and r["target"] in entity_names:
            key = (r["source"], r["type"], r["target"])
            if key not in seen_rels:
                seen_rels.add(key)
                unique_rels.append(r)

    return unique, unique_rels


# ── Store to Neo4j (batched) ─────────────────────────────
def store_kg(nid, entities, rels):
    if not entities:
        return
    with driver.session() as s:
        s.run(
            "UNWIND $ents AS e "
            "MERGE (n:Entity {name: e.name, news_id: $nid}) "
            "SET n.type = e.type",
            ents=entities, nid=nid,
        )
        by_type = {}
        for r in rels:
            by_type.setdefault(r["type"], []).append(r)
        for rtype, group in by_type.items():
            s.run(
                f"UNWIND $rels AS r "
                f"MATCH (a:Entity {{name: r.source, news_id: $nid}}) "
                f"MATCH (b:Entity {{name: r.target, news_id: $nid}}) "
                f"MERGE (a)-[:`{rtype}`]->(b)",
                rels=group, nid=nid,
            )


def get_facts(nid):
    with driver.session() as s:
        rows = s.run(
            "MATCH (a:Entity {news_id:$nid})-[r]->(b:Entity) "
            "RETURN a.name, a.type, type(r), b.name, b.type",
            nid=nid,
        ).data()
    return [
        f"({r['a.name']}:{r['a.type']}) -[{r['type(r)']}]-> ({r['b.name']}:{r['b.type']})"
        for r in rows
    ]


# ── Main ──────────────────────────────────────────────────
df = train_df

with driver.session() as s:
    s.run("MATCH (n) DETACH DELETE n")
    s.run("CREATE INDEX entity_lookup IF NOT EXISTS FOR (n:Entity) ON (n.news_id, n.name)")

t0 = time.time()
for _, row in df.iterrows():
    nid = int(row["id"])
    t1 = time.time()

    ents, rels = extract_kg(row["text"])
    store_kg(nid, ents, rels)
    facts = get_facts(nid)
    n_chunks = len(chunk_text(row["text"]))

    print(f"News {nid}: {n_chunks} chunks, {len(ents)} entities, {len(rels)} rels, {len(facts)} facts ({time.time()-t1:.1f}s)")
    for f in facts[:3]:
        print(f"  {f}")

print(f"\nDone — {len(df)} articles in {time.time()-t0:.1f}s")
driver.close()

/home/simeonk1307/projects/nlp-kg-neo4j/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


News 0: 1 chunks, 5 entities, 3 rels, 3 facts (41.6s)
  (John Folda:PERSON) -[CEO_OF]-> (Fargo Catholic Diocese:ORG)
  (Fargo Catholic Diocese:ORG) -[HEADQUARTERED_IN]-> (North Dakota:LOCATION)
  (Fargo Catholic Diocese:ORG) -[ATTENDED_CONFERENCE]-> (Italy:LOCATION)
News 1: 2 chunks, 10 entities, 5 rels, 5 facts (96.8s)
  (Ralph Mata:PERSON) -[INVESTIGATED_BY]-> (Miami-Dade Police Department:ORG)
  (Miami-Dade Police Department:ORG) -[ARRESTED]-> (Ralph Mata:PERSON)
  (Mata:PERSON) -[WORKED_FOR]-> (Miami-Dade Police Department:ORG)
News 2: 3 chunks, 18 entities, 21 rels, 21 facts (173.0s)
  (Craig Eccleston-Todd:PERSON) -[CAUSED_DEATH_BY_DANGEROUS_DRIVING]-> (Rachel Titley:PERSON)
  (Craig Eccleston-Todd:PERSON) -[DRIVING_WITH_DANGEROUSLY_HIGH_BLOOD_ALCOHOL_CONCENTRATION]-> (Rachel Titley:PERSON)
  (Portsmouth Crown Court:LOCATION) -[TRIAL]-> (Eccleston-Todd:PERSON)
News 3: 2 chunks, 17 entities, 16 rels, 16 facts (110.9s)
  (Vladimir Putin:PERSON) -[RULER_OF]-> (Crimea:LOCATION)
  (Ni

KeyboardInterrupt: 